In [0]:
from pyspark.sql import functions as F

PATH = "/Volumes/vidaplus/ingestion/raw/fact/" 

TABLE_FACT_CLIMA_DIARIO = "vidaplus.bronze.fact_clima_diario"
TABLE_FACT_IBGE_SOCIOECONOMICO_BAIRROS = "vidaplus.bronze.fact_ibge_socioeconomico_bairros"
TABLE_FACT_SINAN_SIVEP_NOTIFICACOES = "vidaplus.bronze.fact_sinan_sivep_notificacoes"
TABLE_FACT_RECURSOS_LEITOS_EQUIPES = "vidaplus.bronze.fact_recursos_leitos_equipes"
TABLE_FACT_RECURSOS_ESTOQUE_INSUMOS = "vidaplus.bronze.fact_recursos_estoque_insumos"

In [0]:
def read_raw_file(file: str):
    return(
        spark.read.format("csv")
        .option("sep", ",")
        .option("header", True)
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .option("encoding", "UTF-8")
        .load(f"{PATH}/{file}")
        .withColumns({
            "_arquivo_origem": F.col("_metadata.file_name"),
            "_ingerido_em": F.current_timestamp(),
        })
    )    

def load_bronze(file: str, table: str):
    return (
        read_raw_file(file).write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(table)
    )    


In [0]:
import traceback

failures = []

# 1. Carga
for file, table_path in [
    ("f2_sinan_sivep_notificacoes.csv", TABLE_FACT_SINAN_SIVEP_NOTIFICACOES),
    ("f3_clima_diario.csv", TABLE_FACT_CLIMA_DIARIO),
    ("f4a_recursos_leitos_equipes.csv", TABLE_FACT_RECURSOS_LEITOS_EQUIPES),
    ("f4b_recursos_estoque_insumos.csv", TABLE_FACT_RECURSOS_ESTOQUE_INSUMOS),
    ("f5_ibge_socioeconomico_bairros.csv", TABLE_FACT_IBGE_SOCIOECONOMICO_BAIRROS),
]:
    try:
        load_bronze(file, table_path)
        count = spark.table(table_path).count()
        print(f"Success | {table_path}: {count:,} registers")
    except Exception:
        print(f"Failed | {table_path}")
        traceback.print_exc()
        failures.append(table_path)

In [0]:
import traceback

table_comments = {
    TABLE_FACT_CLIMA_DIARIO:
    "Bronze - dados climaticos diarios por municipio, como chegou do CSV. "
    "Contem variaveis meteorologicas (temperatura, umidade, precipitacao, vento, pressao, UV) "
    "e dimensoes temporais (estacao do ano, semana epidemiologica). Fonte: INMET-SIM. "
    "Chave: data + cod_municipio_ibge.",
    TABLE_FACT_IBGE_SOCIOECONOMICO_BAIRROS:
    "Bronze - dados socioeconomicos por bairro, como chegou do CSV. "
    "Contem indicadores demograficos, de infraestrutura, saude e vulnerabilidade social. "
    "Fonte: IBGE. Chave: id_bairro + ano.",
    TABLE_FACT_SINAN_SIVEP_NOTIFICACOES:
    "Bronze - notificacoes de agravos do SINAN/SIVEP, como chegou do CSV. "
    "Contem dados de COVID-19 e SRAG com classificacao final, evolucao e hospitalizacao. "
    "Fonte: SIVEP-GRIPE. Chave: id_notificacao.",
    TABLE_FACT_RECURSOS_LEITOS_EQUIPES:
    "Bronze - capacidade operacional diaria por unidade de saude, como chegou do CSV. "
    "Contem leitos clinicos e UTI (operacionais, ocupados, taxa de ocupacao), equipe medica e "
    "de enfermagem (escalados, absenteismo, horas extras) e ventiladores disponiveis. "
    "Chave: data + id_unidade.",
    TABLE_FACT_RECURSOS_ESTOQUE_INSUMOS:
    "Bronze - controle de estoque diario de insumos por unidade de saude, como chegou do CSV. "
    "Contem consumo, recebimento, estoque final, ponto de pedido, dias de cobertura, ruptura "
    "e custo unitario. Chave: data + id_unidade + cod_insumo.",
}

# 2. Comments (after all loads; comment failure does not shut down the load process)
for table_path, comment in table_comments.items():
    try:
        spark.sql(f"COMMENT ON TABLE {table_path} IS '{comment}'")
    except Exception:
        print(f"COMMENT FAILED | {table_path}")
        traceback.print_exc()

In [0]:
validation_failures = []

validation_config = {
    TABLE_FACT_SINAN_SIVEP_NOTIFICACOES: {"expected_cols": 24, "key_cols": ["id_notificacao"], "csv_file": "f2_sinan_sivep_notificacoes.csv"},
    TABLE_FACT_CLIMA_DIARIO: {"expected_cols": 19, "key_cols": ["data", "cod_municipio_ibge"], "csv_file": "f3_clima_diario.csv"},
    TABLE_FACT_RECURSOS_LEITOS_EQUIPES: {"expected_cols": 21, "key_cols": ["data", "id_unidade"], "csv_file": "f4a_recursos_leitos_equipes.csv"},
    TABLE_FACT_RECURSOS_ESTOQUE_INSUMOS: {"expected_cols": 14, "key_cols": ["data", "id_unidade", "cod_insumo"], "csv_file": "f4b_recursos_estoque_insumos.csv"},
    TABLE_FACT_IBGE_SOCIOECONOMICO_BAIRROS: {"expected_cols": 23, "key_cols": ["id_bairro", "ano"], "csv_file": "f5_ibge_socioeconomico_bairros.csv"},
}

for table_path, cfg in validation_config.items():
    try:
        df = spark.table(table_path)
        row_count = df.count()
        actual_cols = len(df.columns)
        checks = []

        if row_count == 0:
            checks.append("Empty Table")
        else:
            if actual_cols != cfg["expected_cols"]:
                checks.append(f"{actual_cols} columns (expected {cfg['expected_cols']})")
            for col in cfg["key_cols"]:
                if df.filter(F.col(col).isNull()).count() == row_count:
                    checks.append(f"column {col} 100% null, file probably corrupted")
            csv_count = read_raw_file(cfg["csv_file"]).count()
            if csv_count != row_count:
                checks.append(f"row count mismatch (CSV={csv_count:,} vs Delta={row_count:,})")

        if checks:
            validation_failures.append(f"{table_path}: {'; '.join(checks)}")
        else:
            print(f"VALID | {table_path}: {row_count:,} register(s), {actual_cols} columns, PK ok, CSV reconciled")
    except Exception as e:
        validation_failures.append(f"{table_path}: {str(e)}")


In [0]:
# 3. Alerta o Job se houver falhas de carga ou validacao
all_failures = failures + validation_failures
if all_failures:
    raise RuntimeError("Falhas:\n" + "\n".join(all_failures))